In [3]:
#import matplotlib.pyplot as plt

In [170]:
import jax.numpy as jnp
import jax
from tqdm import tqdm
#import matplotlib.pyplot as plt
#from matplotlib.animation import FuncAnimation
import numpy as np
#from IPython.display import HTML
#from IPython.display import clear_output

import jax
import jax.numpy as jnp
from jax import random, grad
import optax  # Optimizer library

import numpy as np
import time
from functools import partial
import manim

# Physics Simulation

In [306]:
#GLOBAL CONSTANTS FOR THE CREATES
mu_static = 0.25 #static friction coefficent
mu_kinetic = 0.15 #kinetic friction coefficent
M_min = 1.0 #minimum mass on each vertex
M_total = 15.0 #total mass of the creature #M_max = M_total - 2*M_min if you want!
K_mass = 4.0 #pump constant for the masses 
#The TOTAL mass of the creature is set to 2*M_min + M_max so that one leg can be at max and two at min at any time.

K_spring = 9.0 #8.0 #5.0 #spring constant
#M_dot_max = 10.0 #maximum mass flow rate on pumps, when pump = 1
L_min = 1.6 #minimum spring eq lenth, when muscle = 0
L_max = 2.8 #maximum spring eq length, when muscle = 1

dt = 0.1 #timestep

@jax.jit
def calculate_forces(info):
  #Calcualtes the forces on the three vertices due to:
  # 1. Spring forces
  # 2. Friction
  # Returns the force on each vertex in Shape (batch_size,3,2)
  positions, velocities, masses, pumps, muscles = info
  #Shapes:
  # Vertex Indexed Things:
  #   positions (batch_size, 3, 2)
  #   velocities (batch_size, 3, 2)
  #   masses (batch_size, 3)
  #   pumps (batch_size, 3)
  #     [0,1] valued where 0 represents max shrink, 1 represent max grow
  # Edge indexed things: Edge Order 01, 02, 12
  #   muscles (batch_size, 3)
  #     [0,1] valued  where 0 represents minimum lenght, 1 represents max length


  ### EDGE STUFF

  # Compute edge vectors for all triangles in the batch
  edge_01 = positions[..., 1, :] - positions[..., 0, :]  # Vector from point 0 to point 1
  edge_02 = positions[..., 2, :] - positions[..., 0, :]  # Vector from point 0 to point 2
  edge_12 = positions[..., 2, :] - positions[..., 1, :]  # Vector from point 1 to point 2

   # Compute the current lengths of the edges
  length_01 = jnp.linalg.norm(edge_01, axis=-1)  # Shape: (batch_size,)
  length_02 = jnp.linalg.norm(edge_02, axis=-1)  # Shape: (batch_size,)
  length_12 = jnp.linalg.norm(edge_12, axis=-1)  # Shape: (batch_size,)

  # Normalize edge vectors to get unit directions.
  # make sure lengths are at least epsilon to aviod diveisoon by zero
  epsilon = 1e-8
  dir_01 = edge_01 / jnp.maximum(length_01[..., None],epsilon)  # Shape: (batch_size, 2)
  dir_02 = edge_02 / jnp.maximum(length_02[..., None],epsilon)  # Shape: (batch_size, 2)
  dir_12 = edge_12 / jnp.maximum(length_12[..., None],epsilon)  # Shape: (batch_size, 2)

  # Compute spring forces based on Hooke's law: F = -K * (L - L_eq) * direction

  edge_lengths_eq = L_min + muscles*(L_max - L_min) #equilbirum lengths based on muscle status

  s_force_01 = -K_spring * (length_01 - edge_lengths_eq[..., 0])[..., None] * dir_01  # Shape: (batch_size, 2)
  s_force_02 = -K_spring * (length_02 - edge_lengths_eq[..., 1])[..., None] * dir_02  # Shape: (batch_size, 2)
  s_force_12 = -K_spring * (length_12 - edge_lengths_eq[..., 2])[..., None] * dir_12  # Shape: (batch_size, 2)

  # Initialize forces tensor for all vertices
  # forces is indexed by the vertices now.
  forces = jnp.zeros_like(positions)  # Shape: (batch_size, 3, 2)

  # Accumulate spring forces for each vertex
  forces = forces.at[..., 0, :].add(-s_force_01 - s_force_02)  # Point 0 affected by edges 01 and 02
  forces = forces.at[..., 1, :].add(s_force_01 - s_force_12)   # Point 1 affected by edges 01 and 12
  forces = forces.at[..., 2, :].add(s_force_02 + s_force_12 )   # Point 2 affected by edges 02 and 12

  ###FRICTION FORCES
  # Compute the magnitudes of the velocities for each vertex in the batch
  speed = jnp.linalg.norm(velocities, axis=-1)  # Shape: (batch_size, 3)


  # Determine if each vertex is static (velocity near zero)
  static_epsilon = 1e-8
  is_static = (speed < static_epsilon)  # Shape: (batch_size, 3)
  velocity_directions = jnp.where( is_static[...,None], 0.0, velocities / speed[..., None])  # Safe divide by numbers close to zero. Shape: (batch_size, 3, 2)

  # Compute the magnitudes of the external forces
  force_magnitudes = jnp.linalg.norm(forces, axis=-1)  # Shape: (batch_size, 3)




  # Gravitational acceleration
  g = 10.0  # m/s^2

  # Compute the maximum static friction force magnitudes
  static_friction_max = mu_static * masses * g  # Shape: (batch_size, 3)

  # Compute the kinetic friction force magnitudes
  kinetic_friction_magnitudes = mu_kinetic * masses * g  # Shape: (batch_size, 3)


  forces = jnp.where( jnp.logical_and(is_static[..., None], (force_magnitudes[..., None] <= static_friction_max[..., None]) ),
                     0.0, #if it is static and force is low enough, then set force to zero by static friction
                     forces - kinetic_friction_magnitudes[..., None] * velocity_directions #otherwise apply kinetic friction in direction opposite velocity)
  )

  return forces

@jax.jit
def triangle_update(iter_num, info):
  positions, velocities, masses, pumps, muscles = info
  #Shapes:
  # Vertex Indexed Things:
  #   positions (batch_size, 3, 2)
  #   velocities (batch_size, 3, 2)
  #   masses (batch_size, 3)
  #   pumps (batch_size, 3)
  #     [0,1] valued where 0 represents max shrink, 1 represent max grow
  # Edge indexed things: Edge Order 01, 02, 12
  #   muscles (batch_size, 3)
  #     [0,1] valued  where 0 represents minimum lenght, 1 represents max length


  # Second-Order Runge-Kutta Method (RK2 - Midpoint Method)
  # -------------------------------------------------------
  # 1. Compute acceleration at the initial position and velocity.
  # 2. Estimate velocity and position at the midpoint using half a time step.
  # 3. Compute acceleration at this midpoint.
  # 4. Use the midpoint acceleration to update velocity and position for the full step.
  # - Requires 2 acceleration function calls per step.
  # - Global error is O(dt^2), making it much more accurate than Forward Euler (O(dt)).

  #COMPUTE MASS RATE OF CHANGE BASED ON PUMPS
  #Special rule:
  #if the masss of the leg is less than the minimum leg weight  M_min + dt*M_dot_max, then the creature will override the pump command and pump into that leg regardless of pump status
  # this ensures that the leg mass can never go below M_min
  
  #masses_request = (2.0*pumps - 1.0)
  #masses_request -= jnp.mean( pumps, axis=-1, keepdims=True) #array of entries each in [-1,1] that sums to zero  
  #masses_request += 1.0/3.0 #masses_request sums to 1.0 now
    
  #convert pumps to a fractions, e.g. 0,1,1 -> 0,1/2,1/2. epsilon is there so that (0,0,0) -> (1/3,1/3,1/3)
  #this has the property that all entries are in [0,1] and they sum to 1.
  epsilon = 1e-6
  pumps_fraction = (pumps + epsilon) / jnp.sum( pumps+epsilon, axis=-1, keepdims=True  )

  masses_eq = M_min + pumps_fraction * (M_total - 3.0*M_min) #the **requested** amount of mass to be put at each vertex. 
  #Note that the true thing it has to go for must subtract off the mean
  
  #masses_eq -= jnp.mean(masses_eq, axis=-1,keepdims=True) #now masses_eq sums to 0
  #masses_eq += M_total/3.0 #now mass_eq sums to M_total, so 
    
  #mass_rates = K_mass*(mass_eq-masses) #jnp.where(masses < M_min + dt*M_dot_max, +1.0*M_dot_max,  (2.0*pumps - 1.0)*M_dot_max)  #Shape: (batch_size,3)
  #mass_rates = jnp.clip(mass_rates,-1.0*M_dot_max,+1.0*M_dot_max) #make sure mass rates don't exceed minimum or maximum
    
    #note that mass_rates are guarenteed to sum to zero since sum(mass_eq) = sum(mass) always. 
    
  #make sure the mass rates sum to zero so that total mass of creature is conserved
  #mass_rates -= jnp.mean(mass_rates, axis=-1, keepdims=True) #keepdims=True ensures its still (batch_size,3) after subtracting the mean.

  #RK2 method
#intia
  forces = calculate_forces((positions, velocities, masses, pumps, muscles))
  accelerations = forces / masses[..., None]  # Shape: (3, 2)

  #estiamtes for what is going on at the halfway point
  #masses_half = masses + 0.5*dt*mass_rates #halfway mass update
  #exact solution to the differential equation dm/dt = -k(m - m_eq).
  #. since sum(m_eq)=sum(m)=M_total is fixed, this conserves mass
  masses_half = masses + (masses - masses_eq)*jnp.expm1(-K_mass*0.5*dt)
  #note expm1 is the function exp(x)-1

  positions_half = positions + 0.5*dt*velocities #halfway position update
  velocities_half = velocities + 0.5*dt*accelerations #halfway velocity update

  forces_half = calculate_forces((positions_half, velocities_half, masses_half, pumps, muscles))
  accelerations_half = forces_half / masses_half[..., None]  # Shape: (3, 2)
    
  masses_final = masses + (masses - masses_eq)*jnp.expm1(-K_mass*dt)  
  #Apply the update using the halfway estimates.
  return positions+dt*velocities_half, velocities+dt*accelerations_half, masses_final, pumps, muscles



# Neural Netowrk Brain



In [307]:
# Define a simple fully connected neural network (FCNN) with ReLU activations

def init_params(key, input_size, hidden_size, output_size):
    """Initialize network parameters (weights and biases)"""
    # Weights and biases for the first layer (input to hidden)
    key, subkey = jax.random.split(key)
    W1 = jax.random.normal(subkey, (input_size, hidden_size)) * 0.1
    b1 = jnp.zeros(hidden_size)

    # Weights and biases for the second layer (hidden to output)
    key, subkey = jax.random.split(key)
    W2 = jax.random.normal(subkey, (hidden_size, hidden_size)) * 0.01
    b2 = jnp.zeros(hidden_size)

    # Weights and biases for the second layer (hidden to output)
    key, subkey = jax.random.split(key)
    W3 = jax.random.normal(subkey, (hidden_size, output_size)) * 0.01
    b3 = jnp.zeros(output_size)
    
    #key, subkey = jax.random.split(key)
    #W4 = jax.random.normal(subkey, (hidden_size, output_size)) * 0.1
    #b4 = jnp.zeros(output_size)


    return (W1, b1, W2, b2, W3, b3) #, W4, b4)

@jax.jit
def relu(x):
    """ReLU activation function"""
    return jnp.maximum(0, x)

@jax.jit
def forward(params, x):
    """Feedforward function. Returns the log probs"""
    W1, b1, W2, b2, W3, b3 = params
    first = jnp.dot(x, W1) + b1
    hidden = relu(jnp.dot(first,W2) + b2)  # Hidden layer with ReLU
    #hidden2 = relu(jnp.dot(hidden,W3) + b3)   # Hidden layer with ReLU
    output = jnp.dot(hidden,W3) + b3

    #logits = output - jnp.max(output, axis=-1, keepdims=True)  # Subtract max for numerical stability I think this is auto included in hte jax.nn.log_softamx
    log_probs = jax.nn.log_softmax(output, axis=-1)  # get the log probs
    return log_probs

#@jax.jit
#def softmax(x):
#    """Softmax function for probability distribution"""
#    exp_x = jnp.exp(x - jnp.max(x,axis=-1,keepdims=True))  # Stability trick
#    return exp_x / jnp.sum(exp_x, axis=-1, keepdims=True)


#@jax.jit
#def cross_entropy_loss(logits, targets):
#    """Cross-entropy loss for classification"""
#    probs = softmax(logits)
#    return -jnp.sum(targets * jnp.log(probs + 1e-8),axis=-1)  # Add epsilon #for numerical stability

# Define the neural network model
#@jax.jit
#def model(params, inputs):
#    logits = forward(params, inputs)
#    return softmax(logits)

# Create a function to convert pumps and muscles into one-hot encoded vectors
@jax.jit
def encode_pumps_muscles(pumps, muscles):
    """Encode pumps and muscles as a one-hot vector"""
    # Flatten the 3-bit values for pumps and muscles into a single 6-bit vector
    pumps_flat = (pumps[0] << 2) + (pumps[1] << 1) + pumps[2]
    muscles_flat = (muscles[0] << 2) + (muscles[1] << 1) + muscles[2]
    # We will use the combination of pumps and muscles as a label (from 0 to 63)
    return jnp.array([pumps_flat * 8 + muscles_flat])  # 64 possible configurations

# @jax.jit
# def decode_pumps_muscles(encoded_value):
#     """
#     Decode an integer (0 to 63) into pump and muscle settings.

#     Args:
#         encoded_value (int or jnp.ndarray): A scalar integer or a batch of integers (0 to 63) representing the configuration.

#     Returns:
#         tuple: Two arrays (pumps, muscles), each containing 3 elements (0 or 1) corresponding to the pump and muscle settings.
#     """
#     # Ensure the input is an integer between 0 and 63
#     encoded_value = jnp.clip(encoded_value, 0, 63)

#     # Extract the pumps (first 3 bits)
#     pumps_flat = encoded_value // 8  # Integer division by 8
#     pumps = jnp.array([ (pumps_flat >> i) & 1 for i in range(2, -1, -1)]).T  # Extract the 3 bits and transpose to get the correct shape

#     # Extract the muscles (last 3 bits)
#     muscles_flat = encoded_value % 8  # Get the last 3 bits using modulus 8
#     muscles = jnp.array([ (muscles_flat >> i) & 1 for i in range(2, -1, -1)]).T  # Extract the 3 bits and transpose to get the correct shape

#     # Stack pumps and muscles along the second axis (this will give shape (batch_size, 3))
#     return pumps, muscles

@jax.jit
def decode_pumps_muscles(encoded_value):
    """
    Decode an integer (0 to 63) into pump and muscle settings.

    Args:
        encoded_value (jnp.ndarray): An array of integers (0 to 63) representing the configuration.

    Returns:
        tuple: Two arrays (pumps, muscles), each of shape (..., 3), corresponding to the pump and muscle settings.
    """
    # Ensure the input is an integer between 0 and 63
    encoded_value = jnp.clip(encoded_value, 0, 63)

    # Extract the pumps (first 3 bits)
    pumps_flat = encoded_value // 8  # Integer division by 8
    pumps = jnp.stack([(pumps_flat >> i) & 1 for i in range(2, -1, -1)], axis=-1)

    # Extract the muscles (last 3 bits)
    muscles_flat = encoded_value % 8  # Get the last 3 bits using modulus 8
    muscles = jnp.stack([(muscles_flat >> i) & 1 for i in range(2, -1, -1)], axis=-1)

    return pumps, muscles


@jax.jit
def lengths(points):
  # Define the edge pairs using indices
  edge_indices = jnp.array([[1, 0], [2, 0], [2, 1]])  # Edge pairs: [1,0], [2,0], [2,1]

  # Compute edge vectors by using the indices to select the points
  edges = points[..., edge_indices[:, 0], :] - points[..., edge_indices[:, 1], :]  # Shape: (batch_size, 3, 2)

  # Compute the magnitudes of the edges
  lengths = jnp.linalg.norm(edges, axis=-1)  # Shape: (batch_size, 3)
  return lengths

@jax.jit
def normalize_points(points):
    """
    Normalize the points by subtracting the centroid so that the sum of x and y coordinates is 0.

    Args:
        points (jnp.ndarray): A tensor of shape (batch_size, 3, 2) representing the triangle vertices.

    Returns:
        jnp.ndarray: The normalized points of the same shape (batch_size, 3, 2).
    """
    centroid = jnp.mean(points, axis=-2, keepdims=True)  # Shape: (batch_size, 1, 2)

    normalized_points = points - centroid  # Subtract centroid from all points
    #flattened = normalized_points.reshape(*normalized_points.shape[:-2], 6)
    return normalized_points


@jax.jit
def centroid_y(points):
  centroid = jnp.mean(points[...,1], axis=-1,keepdims=True)  # Shape: (batch_size, 2)
  return centroid



###

input_size = 15 # Flattened size of vertices, lengths and vertex masses
@jax.jit
def get_input(my_points, my_velocities, my_masses, my_pumps, my_muscles): #my_velocities, my_masses
  #return jnp.concatenate([normalize_points(my_points)[...,0],normalize_points(my_points)[...,1],lengths(my_points), my_masses],axis=-1)
  #return jnp.concatenate([lengths(my_points), my_masses],axis=-1)

  #my_centroid(my_points)[1] is the y coordinate of the thing. THis gives it the ability to self correct its y value

  return jnp.concatenate([my_points[...,1],normalize_points(my_points)[...,0],normalize_points(my_points)[...,1],my_pumps, my_muscles],axis=-1)

####


@jax.jit
def brain(params, inputs):
  #input is the concatentated lengths and masses
  # Forward pass to get output probabilities
  logits = forward(params, inputs)
  #output_probs = softmax(logits)
  return logits

# Print the output probability distribution

# Example: jax.random initialization
key = jax.random.PRNGKey(int(time.time()))
hidden_size = 32 # must be same as output size cuz we resnet it to the end
output_size = 64-8  # 64 possible combinations for my_pumps and my_muscles
#-8 so that it does not ever use the state where all pumps are set to +1 (thats the same as setting them all to 0)


# Initialize network parameters
params = init_params(key, input_size, hidden_size, output_size)

# Example input: (my_points, my_velocities, my_masses, my_pumps, my_muscles)
# Here we jax.randomly generate them as an example

#initial velocity and masses
my_velocities_init = jnp.zeros( (3,2) )
my_masses_init = jnp.array([1.0,1.0,1.0])*(M_total)/3.0 #jnp.array([5.0,5.0,5.0])
my_points_init = jnp.array([
    [-1.0, 0.0],   # First vertex
    [1.0, 0.0],    # Second vertex
    [0.0, 1.732]  # Third vertex
]) #1.732 = sqrt(3)

#example points
my_muscles = 0.0*jnp.ones(3)
my_pumps = jnp.array([1.0,1.0,0.0])


# Initialize simulation data
my_points = my_points_init
my_velocities = my_velocities_init
my_masses = my_masses_init


###test it

inputs = get_input(my_points, my_velocities, my_masses, my_pumps, my_muscles)
output_logits = brain(params, inputs) #outputs the log probs directly
print("Output probability distribution over 64 configurations:", output_logits)


Output probability distribution over 64 configurations: [-4.0251713 -4.0250783 -4.024701  -4.0252557 -4.025063  -4.025884
 -4.0253057 -4.02551   -4.0255837 -4.025039  -4.0258293 -4.024767
 -4.0255075 -4.0261173 -4.0254197 -4.0250115 -4.025571  -4.0250626
 -4.0253987 -4.0254498 -4.02506   -4.0253105 -4.025496  -4.025146
 -4.0249133 -4.025043  -4.0249357 -4.02557   -4.0248046 -4.0256495
 -4.0253053 -4.0248265 -4.0257144 -4.0260944 -4.0257    -4.025997
 -4.0262136 -4.024901  -4.02586   -4.02593   -4.025751  -4.0261326
 -4.024853  -4.0248275 -4.025039  -4.0252223 -4.0250015 -4.025234
 -4.0249276 -4.025235  -4.025468  -4.0252695 -4.025274  -4.025336
 -4.025205  -4.0257244]


# RL Stuff

In [308]:
@jax.jit
def compute_reward_score(my_points):
    """
    Computes the x-coordinate of the centroid for each triangle in the batch.

    Args:
        my_points (jnp.ndarray): A tensor of shape (batch_size, 3, 2) representing the triangle vertices.

    Returns:
        jnp.ndarray: A tensor of shape (batch_size,) representing the x-coordinate of the centroid of each triangle.
    """
    # Compute the x-coordinate of the centroid by averaging the x-coordinates of the 3 vertices
    centroid_x = jnp.mean(my_points[..., 0], axis=-1)  # Shape: (batch_size,)
    centroid_y = jnp.mean(my_points[..., 1], axis=-1)  # Shape: (batch_size,)

    return centroid_x - jnp.abs(centroid_y)


# Sample action and compute log-probabilities simultaneously
@jax.jit
def sample_action_and_log_prob(key, logits, temperature=1.0):
    """Sample actions and compute log-probabilities for the chosen actions."""

    # Apply temperature scaling
    scaled_logits = logits / temperature

    # Calculate the probabilities by applying softmax on the logits
    probs = jax.nn.softmax(scaled_logits, axis=-1)

    # Sample actions using categorical distribution
    sample = jax.random.categorical(key, scaled_logits)

    # Get the log-probabilities of the chosen actions
    log_prob = jnp.log(probs[jnp.arange(probs.shape[0]), sample])

    return sample, log_prob

# Sample action and compute log-probabilities simultaneously
@jax.jit
def sample_action(key, logits, temperature=1.0):
    """Sample actions and compute log-probabilities for the chosen actions."""

    # Apply temperature scaling
    scaled_logits = logits / temperature

    # Calculate the probabilities by applying softmax on the logits
    #probs = jax.nn.softmax(scaled_logits, axis=-1)

    # Sample actions using categorical distribution
    sample = jax.random.categorical(key, scaled_logits)

    # Get the log-probabilities of the chosen actions
    #log_prob = jnp.log(probs[jnp.arange(probs.shape[0]), sample])

    return sample #, log_prob



# Intialize Params

# Intialize Optimizer

# Main Training Loop

In [309]:
#brain parameters
num_physics_per_brain_step = 50


batch_size = 100 #batch size per initialziation (so 64x this per run)
my_temp = 10.0 #temperature for sampling words

#create all 64 possible initalizations by applying all 64 words for 1 brain cycle

num_inits = 64-8 #-8 to avoid pump settings of all +1
@jax.jit
def create_inits(key):
  key, subkey = random.split(key)
  theta = random.uniform(subkey, shape=(), minval=0, maxval=2 * jnp.pi)

  # Define the 2D rotation matrix
  rotation_matrix = jnp.array([
      [jnp.cos(theta), -jnp.sin(theta)],
      [jnp.sin(theta),  jnp.cos(theta)]
  ])

  # Apply the rotation
  rotated_init = jnp.dot(my_points_init, rotation_matrix.T)  # Transpose rotation matrix for proper shape


  my_points = jnp.tile(rotated_init, (num_inits , 1, 1))  # Shape: (64, 3, 2)
  my_velocities = jnp.tile(my_velocities_init, (num_inits, 1, 1))  # Shape: (batch_size, 3, 2)
  my_masses = jnp.tile(my_masses_init, (num_inits, 1))  # Shape: (batch_size, 3)
  sample = jnp.arange(num_inits) #jnp.ones(num_inits,dtype=int) #all same initialization! #
  my_pumps, my_muscles = decode_pumps_muscles(sample)
  #run one step of brain activity
  (my_points, my_velocities, my_masses, my_pumps, my_muscles) = jax.lax.fori_loop(0, num_physics_per_brain_step, triangle_update, (my_points, my_velocities, my_masses, my_pumps, my_muscles))
  my_points -= jnp.mean(my_points, axis=-2, keepdims=True) #center them all
  return my_points, my_velocities, my_masses, my_pumps, my_muscles

#create the info for the initalization. shape is (64,3,2) (64,3,2), (64,3),(64,3),(64,3) etc


@partial(jax.jit,static_argnames=["num_brain_steps"])
def create_episodes(params,key,num_brain_steps,my_temp):
  '''Creates a bunch of episodes and returns the action, input and reward histories'''
  action_history = jnp.zeros((num_brain_steps,batch_size,num_inits),dtype=int) #record which word is being used
  input_history = jnp.zeros((num_brain_steps,batch_size,num_inits,input_size)) #record the inputs to the neural networks
  reward_history = jnp.zeros((num_brain_steps,batch_size,num_inits)) #record the rewards

  key, subkey = random.split(key)
  info_init = create_inits(subkey)

  #action_history = action_history.at[0].set(jnp.tile(jnp.arange(num_inits)[None, :], (batch_size, 1)))
  my_points = jnp.tile(info_init[0], (batch_size, 1, 1,1))  # Shape: (batch_size, 3, 2)
  my_velocities = jnp.tile(info_init[1], (batch_size, 1, 1,1))  # Shape: (batch_size, 3, 2)
  my_masses = jnp.tile(info_init[2], (batch_size, 1,1))  # Shape: (batch_size, 3)
  my_pumps = jnp.tile(info_init[3], (batch_size, 1,1))  # Shape: (batch_size, 3)
  my_muscles = jnp.tile(info_init[4], (batch_size, 1,1))  # Shape: (batch_size, 3)

  for brain_step in range(num_brain_steps):

    inputs = get_input(my_points, my_velocities, my_masses, my_pumps, my_muscles)
    input_history = input_history.at[brain_step].set(inputs)


    #Feed input into the brain and sample the muscles+pumps word
    log_probs = forward(params, inputs)
    key, subkey = jax.random.split(key)
    sample = jax.random.categorical(subkey, log_probs/my_temp) # sample_action(subkey, log_probs, temperature=my_temp)
    action_history = action_history.at[brain_step].set(sample)

    #Decode word to get pump and muscle values, and then do physics simulation
    my_pumps, my_muscles = decode_pumps_muscles(sample)
    (my_points, my_velocities, my_masses, my_pumps, my_muscles) = jax.lax.fori_loop(0, num_physics_per_brain_step, triangle_update, (my_points, my_velocities, my_masses, my_pumps, my_muscles))

    #calcualte the reward here.
    reward_history = reward_history.at[brain_step].set(compute_reward_score(my_points))

  return input_history, action_history, reward_history

In [296]:
@jax.jit
def policy_loss(params, inputs, actions, returns): #NOTE this assumpes actions are shape (batch_number,64) makes sure to flatten here!
      log_probs = forward(params, inputs)  # Shape: (batch_size,)
      log_action_probs = log_probs[jnp.arange(actions.shape[0]), actions]  # Correct batch-wise indexing
      return -jnp.mean(log_action_probs * returns)  # Scalar loss

# Compute gradients
loss_gradient_fn = jax.jit(jax.grad(policy_loss))


@partial(jax.jit,static_argnames=["num_brain_steps"])
def single_iteration(params, key, optimizer_state, num_brain_steps, my_temp):
    """Performs a single iteration of training and updates parameters."""
    key, subkey = jax.random.split(key)

    # Create episodes
    input_history, action_history, reward_history = create_episodes(params, subkey, num_brain_steps, my_temp)


    last_reward = jnp.mean(reward_history[-1])

    # Flatten episodes
    f_input_history = input_history.reshape((num_brain_steps, batch_size * num_inits, input_size))
    f_action_history = action_history.reshape((num_brain_steps, batch_size * num_inits))

    epsilon = 1e-7
    group_normed_reward_history = (reward_history - jnp.mean(reward_history,axis=-1,keepdims=True )) / (jnp.std(reward_history,axis=-1,keepdims=True) + epsilon )
    f_reward_history = group_normed_reward_history.reshape((num_brain_steps, batch_size * num_inits))



    # Accumulate gradients
    accumulated_grads = None

    for t in range(num_brain_steps):
        # Compute the gradient at time step t
        my_return = f_reward_history[-1] #- (0 if t == 0 else f_reward_history[t-1])
        loss_gradient = loss_gradient_fn(params, f_input_history[t], f_action_history[t], my_return)

        # Accumulate gradients over time steps
        if accumulated_grads is None:
            accumulated_grads = loss_gradient
        else:
            accumulated_grads = jax.tree_util.tree_map(lambda a, b: a + b, accumulated_grads, loss_gradient)

    accumulated_grads = jax.tree_util.tree_map(lambda g: g / num_brain_steps, accumulated_grads)

    # Apply optimizer update
    updates, new_optimizer_state = optimizer.update(accumulated_grads, optimizer_state)
    new_params = optax.apply_updates(params, updates)

    return new_params, key, new_optimizer_state, last_reward


def run_training_loop(params, key, optimizer_state, num_iters, num_brain_steps, my_temp=10):
    """Runs the reinforcement learning training loop with JAX JIT optimization."""

    final_reward = np.zeros(num_iters)

    for iter in tqdm(range(num_iters)):
        params, key, optimizer_state, final_reward[iter] = single_iteration(params, key, optimizer_state, num_brain_steps, my_temp)

        interval = (num_iters//10)
        if (iter + 1) % interval == 0:
            print(f"{iter=}, Avg:{np.mean(final_reward[iter+1-interval:iter+1])/num_brain_steps:.3f}, SD:{np.std(final_reward[iter+1-interval:iter+1])/num_brain_steps:.3f}")

    return params, key, optimizer_state, final_reward


In [311]:

# Initialize network parameters
params = init_params(key, input_size, hidden_size, output_size)


In [312]:
# Initialize optimizer (e.g., Adam)
optimizer = optax.adam(learning_rate=1e-2)
optimizer_state = optimizer.init(params)


In [356]:
for num_brain_steps in range(10,12): #num_brain_steps = 11
#num_brain_steps = 5
    my_temp = 50
    num_iters = 1000
    params, key, optimizer_state, final_rewards = run_training_loop(params, key, optimizer_state, num_iters=num_iters, num_brain_steps=num_brain_steps, my_temp=my_temp)
    print(f"{num_brain_steps=}, {np.mean(final_rewards)/num_brain_steps=}")

 10%|████                                    | 100/1000 [04:04<35:34,  2.37s/it]

iter=99, Avg:0.889, SD:0.024


 20%|████████                                | 200/1000 [08:00<32:51,  2.46s/it]

iter=199, Avg:0.890, SD:0.031


 30%|████████████                            | 300/1000 [12:20<30:31,  2.62s/it]

...

iter=899, Avg:0.913, SD:0.032


100%|███████████████████████████████████████| 1000/1000 [41:26<00:00,  2.49s/it]


iter=999, Avg:0.913, SD:0.036
num_brain_steps=10, np.mean(final_rewards)/num_brain_steps=np.float64(0.8996905134677886)


  1%|▎                                         | 6/1000 [00:05<16:09,  1.03it/s]


ValueError: setting an array element with a sequence.

In [373]:
# Convert JAX arrays to NumPy and save as `.npz`
np.savez("jax_params_avg_score_0913_Feb4.npz", *[np.array(p) for p in params])  # Saves as positional arguments

In [227]:
for temp in range(5):
    pass
print(temp)

4


In [357]:


def create_plotting_episode(params,key,starting_word,num_frames_per_brain,num_brain_steps,my_temp=10):
  '''Creates a bunch of episodes and returns the action, input and reward histories'''
  action_history = jnp.zeros((num_brain_steps),dtype=int) #record which word is being used
  my_points_history = jnp.zeros((num_frames_per_brain*num_brain_steps,3,2)) #record the inputs to the neural networks
  my_masses_history = jnp.zeros((num_frames_per_brain*num_brain_steps,3))


  key, subkey = random.split(key)
  info_init = create_inits(subkey)

  my_points = info_init[0][starting_word]
  my_velocities = info_init[1][starting_word]
  my_masses = info_init[2][starting_word]
  my_pumps = info_init[3][starting_word]
  my_muscles = info_init[4][starting_word]


  for brain_step in range(num_brain_steps):



    inputs = get_input(my_points, my_velocities, my_masses, my_pumps, my_muscles)


    #Feed input into the brain and sample the muscles+pumps word
    log_probs = forward(params, inputs)
    key, subkey = jax.random.split(key)
    sample = jax.random.categorical(subkey, log_probs/my_temp) # sample_action(subkey, log_probs, temperature=my_temp)
    action_history = action_history.at[brain_step].set(sample)

    #Decode word to get pump and muscle values, and then do physics simulation
    my_pumps, my_muscles = decode_pumps_muscles(sample)

    for frame in range(num_frames_per_brain):
      my_points_history = my_points_history.at[frame + num_frames_per_brain*brain_step].set(my_points)

      my_masses_history = my_masses_history.at[frame + num_frames_per_brain*brain_step].set(my_masses)

      (my_points, my_velocities, my_masses, my_pumps, my_muscles) = jax.lax.fori_loop(0, num_physics_per_brain_step//num_frames_per_brain, triangle_update, (my_points, my_velocities, my_masses, my_pumps, my_muscles))


  return action_history, my_points_history, my_masses_history


In [17]:
from manim import *

In [39]:
## Global Variables and Stuff ##
config.media_embed = True
DELAY = 0.1

# latex preamble
texPre = TexTemplate()
texPre.add_to_preamble(r"""
    \usepackage{amsmath}
    \usepackage{amssymb}
    \newcommand{\E}{\mathbb{E}}
    \newcommand{\vect}[1]{\mathbf{#1}}
""") 

# tex color dictionary
E_color = GREY_B
t2cD = {
    r"\E":E_color,
}

# font sizes
my_fs = 75

Starting Word: 5


In [362]:
def sample_to_string(sample):
            wordlist = ["ant", 
                        "bat", "bear", "bee", "beaver",
                        "cat", "crab", "cow", "caribou",
                        "dog", "duck", "deer", 
                        "elk",  
                        "fox", "frog", 
                        "goat", "goose",
                        "hawk", "horse",
                        "iguana", 
                        "jaguar", "jackal", 
                        "koala", 
                        "llama", "lynx", "loon",
                        "mouse", "mole", "moose",
                        "newt", "narwhale", 
                        "owl", "orca", "otter", 
                        "pig", "panda", "parrot",
                        "quail", 
                        "rabbit", "rat", 
                        "seal", "shark", "swan","skunk", 
                         "toad", "turtle", "tiger", "turkey",
                        "urchin", 
                        "viper", "vulture", 
                        "whale", "wolf", "walrus",
                        "yak","zebra"]
            #print(len(wordlist))
            wordlist = [entry.capitalize() for entry in wordlist]
            return wordlist[sample]

In [243]:
       
class TriCreature(VGroup):
    def __init__(self, vertex_coords, v_radius=0.2, e_width=8, s_width=2):
        super().__init__()
        # Create three movable vertices (dots)
        self.v = VGroup(*[Dot(coord, radius=v_radius).set_stroke(color=WHITE,width=s_width)
                          for coord in vertex_coords])
        self.v.set_z_index(0.01)

        # Create edges as lines initially connected to the vertices
        self.e = VGroup(*[
            Line(self.v[0].get_center(), self.v[1].get_center()).set_stroke(width=e_width),
            Line(self.v[1].get_center(), self.v[2].get_center()).set_stroke(width=e_width),
            Line(self.v[2].get_center(), self.v[0].get_center()).set_stroke(width=e_width)
        ])

        # Updater function to keep edges connected to vertices
        
        #self.add_edge_updater()
        self.scale = 1
        self.offx = 0
        self.offy = 0
        self.add(self.v, self.e)
    def generate_target(self):
        # Call the original generate_target method to generate the base target
        super().generate_target()

        # Now modify the target directly
        self.target.v = self.target[0]  # Change the color of the target
        self.target.e = self.target[1]  # Move the target up

        # You can add more modifications to self.target as needed

        return self.target
    
    def plant(self):
        #to be called when the object is in the initial time=0 orientation AND moved ot the correct place on the string
        #set the offsets to the current position so that future updates move it accordingly
        self.offx = self.get_x()
        self.offy = self.get_y()
    
    def add_edge_updater(self):
        def update_edges(_):
            self.e[0].put_start_and_end_on(self.v[0].get_center(), self.v[1].get_center())
            self.e[1].put_start_and_end_on(self.v[1].get_center(), self.v[2].get_center())
            self.e[2].put_start_and_end_on(self.v[2].get_center(), self.v[0].get_center())
       
        #self.e.add_updater(update_edges)
        

    def set_pumps_muscles(self, pumps, muscles):
        vertex_color = ['#308EB3','#B3144B'] #['#45caff','#ff1b6b'] #['#ffdd55','#663399'] #[BLUE,RED]
        edge_color = ['#45caff','#ff1b6b'] #['#fcc602','#4c257e'] #[BLUE,RED]\
        # Vertex colors (Dark Red and Dark Blue)
        #vertex_color = ['#B22222', '#1E3A5F']  # [Red (Dark), Blue (Dark)]

        # Edge colors (Light Red and Light Blue)
        #edge_color = ['#FF7F7F', '#ADD8E6']  # [Red (Light), Blue (Light)]
        for i, pump_val in enumerate(pumps):
            self.v[i].set_fill(vertex_color[pump_val])
        for i, muscle_val in enumerate(muscles):
            self.e[i].set_color(edge_color[muscle_val])
            
    def set_points(self,points):
        #input points as a (3,2) and set the vertices
        for i in range(3):
            self.v[i].set_x(self.offx + self.scale*points[i,0])
            self.v[i].set_y(self.offy + self.scale*points[i,1])
        for i, (v_start_ix, v_end_ix) in enumerate([(0, 1), (0, 2), (1, 2)]):
            self.e[i].put_start_and_end_on(self.v[v_start_ix].get_center(),self.v[v_end_ix].get_center())
        #self.add_edge_updater()
        
    def set_masses(self,masses):
        for i, mass in enumerate(masses):
            R = 0.15
            radius = np.sqrt(mass)*R*self.scale
            self.v[i].set_width(radius)


In [26]:
from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.gtts import GTTSService

In [369]:
#my_temp = 100.0
#num_brain_steps = 20
key = jax.random.PRNGKey(int(time.time()))
key, subkey = random.split(key)  # Ensure reproducibility
num_frames_per_brain = 5
starting_word = random.randint(subkey, shape=(), minval=0, maxval=num_inits).item()
print(f"Starting Word: {starting_word}")
action_history, my_points_history, my_masses_history = create_plotting_episode(params, subkey, starting_word, num_frames_per_brain,num_brain_steps, my_temp)


Starting Word: 12


In [372]:
%%manim -qh -v CRITICAL --disable_caching MyScene
# %%manim_slides -ql -v WARNING MyScene --manim-slides controls=true
# Choose to do regular manim or manim-slides by choosing the correct jupyter magic. Must be first line.


 
#num_brain_steps = 10

class MyScene(VoiceoverScene): # change to MyScene(Slide) for manim-slides
    def pause(self):
        self.wait(DELAY)
        #self.next_slide() #comment in for manim-slides
        
    def construct(self):
        
        self.set_speech_service(GTTSService(lang="en", tld="com"))
        #self.set_speech_service(GTTSService(lang="en", tld="com"))
        #with self.voiceover(text="here is the first text") as tracker:
        #    self.play(Create(Circle()))
        #self.next_section(skip_animations=True) #comment out for manim-slides
        #self.pause()
        #return 0
        
        
        ###
        self.next_section() #comment out for manim-slied
        
        vertex_coords = [LEFT * 2 + DOWN, RIGHT * 2 + DOWN, UP * 2]
        tri = TriCreature(vertex_coords)
        
        # Add triangle to the scene
        
        
        

        tri.set_pumps_muscles(*decode_pumps_muscles(starting_word))
        tri.scale =  (config.frame_width/2.0 - 1.0) *( 2.0/np.mean(my_points_history[-1,:,0]) )
        tri.set_points(my_points_history[0])
        #print("ok")
        #return 0
        tri.set_masses(my_masses_history[0])
        tri.shift(LEFT*tri.scale*np.mean(my_points_history[-1,:,0])/2.0 ) #move it over so that its path is centered around x=0
        tri.plant()
        self.add(tri)
        print(*[int(my_pumps[i]) for i in range(3)], *[int(my_muscles[i]) for i in range(3)])
        
        
        word_speed = 2.5
        fade_in_rate = 2.0
        def update_fade_in(mob,dt):
            old_opacity = mob.get_fill_opacity() #()
            #if old_opacity:
            new_opacity = min(old_opacity + dt * fade_in_rate, 1)  # Increase opacity gradually
            #else:
            #    new_opacity = 0.0
            mob.set_opacity(new_opacity)
        def update_move_to_target(mob,dt):
            # Get the current position
            #current_position = mob.get_center()

            # Calculate the direction to move towards the target location
            direction = mob.target.get_center() - mob.get_center() #current_position
            
            distance_to_move = word_speed * dt

            # If the distance to the target is smaller than the distance to move, move directly to the target
            if np.linalg.norm(direction) <= distance_to_move:
                mob.move_to(mob.target.get_center())
            else:
                # Otherwise, move towards the target with constant speed
                mob.shift(direction / np.linalg.norm(direction) * distance_to_move)


            # Update the position by a fraction of the distance (based on dt)
            #mob.shift(direction * dt * word_speed)  # Adjust the multiplier for speed

     
        
        
        my_word_list = []
        for brain_ix in range(num_brain_steps):
            sample = action_history[brain_ix]
            #pumps,muscles = decode_pumps_muscles(sample)
            text = sample_to_string(sample) #pumps_muscles_to_string(pumps,muscles) #"".join(str(num) for num in list(pumps) + list(muscles))
            my_word_list.append(text+" ")
        monol_words = Tex(*my_word_list)
            
        #word = Text(text)
        #word.add_updater(update_fade_in)
        #word.add_updater(update_move_to_target)
        #monol_words.add(word)

        #monol_words.arrange(RIGHT)
        # Get the width of the screen (frame width)
        buff = 0.3
        screen_width = config.frame_width - 2*buff

        # Get the width of the text object
        text_width = monol_words.get_width()

        # Calculate the scaling factor to fit the text to the screen width
        scaling_factor = screen_width / text_width

        # Scale the text
        monol_words.scale(scaling_factor)
        monol_words.to_corner(UL,buff=buff)
        
        for brain_ix in range(num_brain_steps):
            
            monol_words[brain_ix].generate_target()
            monol_words[brain_ix].add_updater(update_fade_in)
            monol_words[brain_ix].add_updater(update_move_to_target)
            
        #self.add(monol_words)
        
        #if brain_ix % 2 == 0:
        #    word.add_updater(update_word_up)
        #elif brain_ix % 2 ==1:
            
        #    
            
        
        
        
        #self.add(*monol_words)
        
        tri.generate_target()
        
        for brain_ix in range(num_brain_steps):
            sample = action_history[brain_ix]
            tri.target.set_pumps_muscles(*decode_pumps_muscles(sample))
            ix = brain_ix*num_frames_per_brain
            monol_words[brain_ix].set_x( tri.offx + tri.scale*np.mean(my_points_history[ix,:,0] )  )
            if brain_ix % 2 == 0:
                monol_words[brain_ix].set_y( tri.offy + tri.scale*np.max(my_points_history[ix,:,1] )  )
            elif brain_ix % 2 == 1:
                monol_words[brain_ix].set_y( tri.offy + tri.scale*np.max(my_points_history[ix,:,1] )  )
            monol_words[brain_ix].set_opacity(0.0)
            self.add(monol_words[brain_ix])
            
            #with self.voiceover(text=sample_to_string(sample)) as tracker:
            for frame_num in range(num_frames_per_brain):
                ix = frame_num + brain_ix*num_frames_per_brain
                tri.target.set_points(my_points_history[ix])
                tri.target.set_masses(my_masses_history[ix])
                self.play(MoveToTarget(tri),rate_func=linear, run_time=0.20)

        #for i in range(num_brain_steps*num_frames_per_brain):
        #    anim_list = []
        #    if i % num_frames_per_brain == 0:
                
                #self.play(monol_words[parity].animate.shift(2*UP*(1-2*parity)))
                
            
        #    my_text = monol_words[brain_ix]
        #    with self.voiceover(text=my_text) as tracker:
                

        self.wait(3)
        
       

Manim Community v0.19.0

/var/folders/ny/ts_9hspn65bfvpq98jt4ybmh0000gn/T/ipykernel_63821/2176132653.py:76: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  self.v[i].set_width(radius)
<string>:98: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.


1 1 0 0 0 0


In [226]:
print(num_brain_steps)

20


In [1]:
x=2

In [2]:
print(x)

2


In [155]:
wordlist = ["alligator",
                        "apron",
                        "bear",
                        "balloon",
                        "cat",
                        "camera",
                        "deer",
                        "dress",
                        "eagle",
                        "egg",
                        "fox",
                        "flowers",
                        "giraffe",
                        "glasses",
                        "hedgehog",
                        "hat",
                        "iguana",
                        "ice", "cream"
                        "jaguar",
                        "jump", "rope"
                        "koala",
                        "kite",
                        "llama",
                        "lemonade",
                        "mouse",
                        "muffin",
                        "newt",
                        "newspaper",
                        "owl",
                        "overalls",
                        "pig",
                        "pie",
                        "quail",
                        "quilt",
                        "rabbit",
                        "rain", "boots",
                        "squirrel",
                        "socks",
                        "tiger",
                        "telephone",
                        "urchin",
                        "vampire", "bat",
                        "violin",
                        "whale",
                        "waffles",
                        "x-ray", "fish",
                        "xylophone",
                        "yak",
                        "yarn",
                        "zebra",
                        "zipper", "unusual", "animal"]
print(len(wordlist))

56


In [220]:
#my_temp = 100.0
num_brain_steps = 20
key = jax.random.PRNGKey(int(time.time()))
key, subkey = random.split(key)  # Ensure reproducibility
num_frames_per_brain = 25

starting_word = random.randint(subkey, shape=(), minval=0, maxval=num_inits).item()
print(f"Starting Word: {starting_word}")

my_pumps, my_muscles = decode_pumps_muscles(starting_word)
print(*[int(my_pumps[i]) for i in range(3)], *[int(my_muscles[i]) for i in range(3)])

def create_plotting_episode(params,key,starting_word,num_frames_per_brain,my_temp=10):
  '''Creates a bunch of episodes and returns the action, input and reward histories'''
  action_history = jnp.zeros((num_brain_steps),dtype=int) #record which word is being used
  my_points_history = jnp.zeros((num_frames_per_brain*num_brain_steps,3,2)) #record the inputs to the neural networks
  my_masses_history = jnp.zeros((num_frames_per_brain*num_brain_steps,3))


  key, subkey = random.split(key)
  info_init = create_inits(subkey)

  my_points = info_init[0][starting_word]
  my_velocities = info_init[1][starting_word]
  my_masses = info_init[2][starting_word]
  my_pumps = info_init[3][starting_word]
  my_muscles = info_init[4][starting_word]


  for brain_step in range(num_brain_steps):



    inputs = get_input(my_points, my_velocities, my_masses, my_pumps, my_muscles)


    #Feed input into the brain and sample the muscles+pumps word
    log_probs = forward(params, inputs)
    key, subkey = jax.random.split(key)
    sample = jax.random.categorical(subkey, log_probs/my_temp) # sample_action(subkey, log_probs, temperature=my_temp)
    action_history = action_history.at[brain_step].set(sample)

    #Decode word to get pump and muscle values, and then do physics simulation
    my_pumps, my_muscles = decode_pumps_muscles(sample)

    for frame in range(num_frames_per_brain):
      my_points_history = my_points_history.at[frame + num_frames_per_brain*brain_step].set(my_points)

      my_masses_history = my_masses_history.at[frame + num_frames_per_brain*brain_step].set(my_masses)

      (my_points, my_velocities, my_masses, my_pumps, my_muscles) = jax.lax.fori_loop(0, num_physics_per_brain_step//num_frames_per_brain, triangle_update, (my_points, my_velocities, my_masses, my_pumps, my_muscles))


  return action_history, my_points_history, my_masses_history

action_history, my_points_history, my_masses_history = create_plotting_episode(params, subkey, starting_word, num_frames_per_brain, my_temp)


# Set up the figure and axis for the animation
fig, ax = plt.subplots()
sc = ax.scatter([], [], s=[], c=[], cmap="coolwarm", edgecolors="k", vmin=0, vmax=1)
lines = [ax.plot([], [], lw=3, alpha=0.8)[0] for _ in range(3)]  # One for each edge

ax.set_xlim(-5, 10)
ax.set_ylim(-5, 5)

def update_frame(i):
    global key, my_points, my_velocities, my_masses, my_pumps, my_muscles
    #print(i)
    if i % num_frames_per_brain == 0:

      plt.title(f"Iteration: {i//num_frames_per_brain}")
      sample = action_history[i//num_frames_per_brain]
      print(i//num_frames_per_brain, sample)
      my_pumps, my_muscles = decode_pumps_muscles(sample)
      print(*[int(my_pumps[i]) for i in range(3)], *[int(my_muscles[i]) for i in range(3)])

    my_points = my_points_history[i]
    my_masses = my_masses_history[i]
    # Extract x and y coordinates of vertices
    x, y = my_points[:, 0], my_points[:, 1]

    # Update vertex sizes (proportional to mass)
    sizes = jnp.sqrt(my_masses)*50  # Scale appropriately for visibility

    # Update vertex colors (based on my_pumps)
    vertex_colors = my_pumps

    # Update edges
    for j, (p1, p2) in enumerate([(0, 1), (0,2), (1, 2)]):
        lines[j].set_data([x[p1], x[p2]], [y[p1], y[p2]])

        # Set edge color based on my_muscles
        color_value = 1.0*my_muscles[j]
        lines[j].set_color(plt.cm.coolwarm(color_value))  # Apply colormap

    # Update scatter plot for vertices
    sc.set_offsets(my_points)
    sc.set_sizes(sizes)
    sc.set_array(vertex_colors)  # Apply colormap to vertices

# Create the animation using FuncAnimation
ani = FuncAnimation(fig, update_frame, frames=num_brain_steps*num_frames_per_brain, interval=30, blit=False)

# Display the animation in Google Colab using HTML
HTML(ani.to_html5_video())

Starting Word: 13
0 0 1 1 0 1


NameError: name 'plt' is not defined

In [65]:
print(params)

(Array([[-0.27296576, -0.46061355,  0.04169488, -0.36242864, -0.3471872 ,
        -0.48391998,  0.28968865,  0.25149524, -0.3575242 , -0.63247865,
         0.38898498, -0.6672709 , -0.30151194,  0.04439636,  0.1427174 ,
        -0.33091515,  0.08966385,  0.5785801 ,  0.07253598,  0.37162   ,
         0.2824757 ,  0.342665  , -0.35007244,  0.40308657, -0.3274003 ,
         0.2849936 , -0.2936573 , -0.5006998 ,  0.5285086 ,  0.3700211 ,
         0.08336692, -0.42842743, -0.28524774,  0.5210624 ,  0.0494685 ,
         0.45668945, -0.29190525, -0.57480663,  0.24136539,  0.02139003,
         0.18118292, -0.51912373,  0.55307835, -0.44834113,  0.3048904 ,
        -0.47103193,  0.19091001, -0.184779  , -0.37031338,  0.36122125,
         0.50917345,  0.5781808 , -0.3825486 , -0.25554886,  0.3431543 ,
         0.4064454 ,  0.23680349, -0.44552562, -0.19940375,  0.2820362 ,
         0.36649758, -0.2596164 , -0.47699994, -0.34663582],
       [ 0.4084216 ,  0.19777362, -0.14479642,  0.524217  ,  0

In [66]:
num_iters = 1000
final_reward = np.zeros(num_iters)
for iter in tqdm(range(num_iters)):
  key,subkey = jax.random.split(key)
  # Create episodes
  input_history, action_history, reward_history = create_episodes(params,subkey)
  final_reward[iter] = jnp.mean(reward_history[-1])
  if (iter+1) % 100 == 0:
    print(f"{iter=}, Avg:{np.mean(final_reward[iter+1-100:iter+1]):.2f}, SD:{np.std(final_reward[iter+1-100:iter+1]):.2f}")

  # Flatten episodes
  f_input_history = input_history.reshape((num_brain_steps,batch_size*num_inits,input_size))
  f_action_history = action_history.reshape((num_brain_steps,batch_size*num_inits))
  f_reward_history = reward_history.reshape((num_brain_steps,batch_size*num_inits))

  # Accumulate gradients
  accumulated_grads = None

  for t in range(num_brain_steps):
    # Compute the gradient at time step t
    if t==0:
      my_return = f_reward_history[-1] - 0
    else:
      my_return = f_reward_history[-1] - f_reward_history[t]
    loss_gradient = loss_gradient_fn(params, f_input_history[t], f_action_history[t], my_return)

    # Accumulate gradients over time steps
    if accumulated_grads is None:
        accumulated_grads = loss_gradient  # First step: Initialize
    else:
        accumulated_grads = jax.tree_util.tree_map(lambda a, b: a + b, accumulated_grads, loss_gradient)  # Sum gradients (need weird function cuz its a tuple)

  accumulated_grads = jax.tree_util.tree_map(lambda g: g / num_brain_steps, accumulated_grads)

  # Apply a **single** optimizer update after accumulating gradients
  updates, new_optimizer_state = optimizer.update(accumulated_grads, optimizer_state)
  params = optax.apply_updates(params, updates)

  9%|▉         | 93/1000 [00:55<09:05,  1.66it/s]


ValueError: setting an array element with a sequence.

In [ ]:
action_history[-1]

In [ ]:
create_inits()

In [ ]:
# Init conditions
my_muscles = 0.0*jnp.ones(3)
my_pumps = jnp.array([1.0,1.0,0.0])
my_temp = 50.0

batch_size = 40_000

# Initialize simulation data
my_points = my_points_init
my_velocities = my_velocities_init
my_masses = my_masses_init


num_training_iters = 10000

#gamma = 0.8 #0.99  # Discount factor


accumulated_grads = None


avg_distances = np.zeros((num_training_iters))

std_distances = np.zeros((num_training_iters))

key = jax.random.PRNGKey(int(time.time()))
r = (L_min + L_max)/2.0
key, subkey = random.split(key)  # Ensure reproducibility
a0 = random.uniform(subkey, shape=(num_training_iters,), minval=0.5*jnp.pi/3.0, maxval= 1.5*jnp.pi/3.0)
key, subkey = random.split(key)  # Ensure reproducibility
a1 = random.uniform(subkey, shape=(num_training_iters,), minval=2.5*jnp.pi/3.0, maxval= 3.5*jnp.pi/3.0)
key, subkey = random.split(key)  # Ensure reproducibility
a2 = random.uniform(subkey, shape=(num_training_iters,), minval=4.5*jnp.pi/3.0, maxval= 5.5*jnp.pi/3.0)

# Compute the corresponding points on the unit circle
x0, y0 = r * jnp.cos(a0), r * jnp.sin(a0)
x1, y1 = r * jnp.cos(a1), r * jnp.sin(a1)
x2, y2 = r * jnp.cos(a2), r * jnp.sin(a2)

# Stack to get shape (num_training_iters, 3, 2)
my_points_init = jnp.stack([jnp.stack([x0, y0], axis=-1),
                            jnp.stack([x1, y1], axis=-1),
                            jnp.stack([x2, y2], axis=-1)], axis=1)




for iters in tqdm(range(num_training_iters)):

    # Replicate the initial state across the batch dimension


  #as = jnp.array([a0, a1, a2])  # Ensure as is (3,)
  #radii = jnp.full((3,), r)  # Ensure r is (3,)

  #my_points_init = jnp.stack([radii * jnp.cos(as), radii * jnp.sin(as)], axis=-1)



  #my_points_init = jnp.array([[-1.0,-1.0],[1.0,-1.0],[0.0,1.0]])

  my_points = jnp.tile(my_points_init[iters], (batch_size, 1, 1))  # Shape: (batch_size, 3, 2)
  my_velocities = jnp.tile(my_velocities_init, (batch_size, 1, 1))  # Shape: (batch_size, 3, 2)
  my_masses = jnp.tile(my_masses_init, (batch_size, 1))  # Shape: (batch_size, 3)


  action_history = np.zeros((num_brain_steps,batch_size),dtype=int)
  input_history = np.zeros((num_brain_steps,batch_size,input_size))
  #reward_history = np.zeros((num_brain_steps,batch_size)) #the rewards will be based on this history
  log_action_taken_probs_history = np.zeros((num_brain_steps,batch_size)) #the log probability of the action we actually took

  for brain_step in range(num_brain_steps):

    #Create input into the brain
    #inputs = jnp.concatenate([lengths(my_points), my_masses],axis=-1)

    inputs = jnp.concatenate([normalize_points(my_points)[...,0],normalize_points(my_points)[...,1],lengths(my_points), my_masses],axis=-1)
    state_history[brain_step] = inputs

    #Feed input into the brain to get muscles+pumps word
    logits = forward(params, inputs)
    #print(jnp.shape(scaled_logits))
    key, subkey = jax.random.split(key)
    #sample, log_prob = sample_action_and_log_prob(key, logits, temperature=my_temp)
    sample = sample_action(subkey, logits, temperature=my_temp)

    action_history[brain_step] = sample
    #log_action_taken_probs_history[brain_step] = log_prob

    #Decode word to get pump and muscle values, and then do physics simulation
    my_pumps, my_muscles = decode_pumps_muscles(sample)
    #print(sample)
    #centroid_before = compute_centroid_x(my_points)

    (my_points, my_velocities, my_masses, my_pumps, my_muscles) = jax.lax.fori_loop(0, num_physics_per_brain_step, triangle_update, (my_points, my_velocities, my_masses, my_pumps, my_muscles))

    #print(my_points[0,0,0])


    #centroid_after = compute_centroid_x(my_points)
    #reward_history[brain_step] = centroid_after - centroid_before

  eps = 1e-8  # Small constant to prevent division by zero
  #reward_history = (reward_history - np.mean(reward_history, axis=-1, keepdims=True)) / (np.std(reward_history, axis=-1, keepdims=True) + eps)

  final_rewards =  2*compute_centroid_x(my_points) - jnp.abs(compute_centroid_y(my_points))
  avg_distances[iters] = jnp.mean(final_rewards)
  std_distances[iters] = jnp.std(final_rewards)
  final_rewards = (final_rewards - avg_distances[iters]) / (std_distances[iters] + eps)

  #reward_history[-1] = final_rewards

  # Perform an update
  #for k in range(num_updates_per_sample):
  #  params, optimizer_state, loss = reinforce_update(
  #      params, action_history, state_history, reward_history, #log_action_taken_probs_history, gamma, optimizer_state, optimizer.update
  #  )




  #for t in range(num_brain_steps):
  #  loss_gradient = loss_gradient_fn(params, inputs, action_history[t, :], final_rewards)

  #  updates, new_optimizer_state = optimizer.update(loss_gradient, optimizer_state)
  #  params = optax.apply_updates(params, updates)
    #updates, new_optimizer_state = optimizer.update(grads, optimizer_state)


    # Compute log probabilities correctly
  def log_probs(params, inputs, actions):
      logits = forward(params, inputs)  # Shape: (batch_size, num_actions)
      log_probs = jax.nn.log_softmax(logits, axis=-1)  # Shape: (batch_size, num_actions)
      return log_probs[jnp.arange(actions.shape[0]), actions]  # Correct batch-wise indexing

  # Compute policy gradient loss (negative log probs * reward-to-go)
  def policy_loss(params, inputs, actions, returns):
      log_action_probs = log_probs(params, inputs, actions)  # Shape: (batch_size,)
      return -jnp.mean(log_action_probs * returns)  # Scalar loss

  # Compute gradients
  loss_gradient_fn = jax.grad(policy_loss)

  accumulated_grads = None
  for t in range(num_brain_steps):
    # Compute the gradient at time step t
    loss_gradient = loss_gradient_fn(params, inputs, action_history[t, :], final_rewards)

    # Accumulate gradients over time steps
    if accumulated_grads is None:
        accumulated_grads = loss_gradient  # First step: Initialize
    else:
        accumulated_grads = jax.tree_util.tree_map(lambda a, b: a + b, accumulated_grads, loss_gradient)  # Sum gradients

  # Normalize the accumulated gradients (optional, but often useful)
  num_steps = num_brain_steps
  accumulated_grads = jax.tree_util.tree_map(lambda g: g / num_steps, accumulated_grads)

  # Apply a **single** optimizer update after accumulating gradients
  updates, new_optimizer_state = optimizer.update(accumulated_grads, optimizer_state)
  params = optax.apply_updates(params, updates)

  # Update the model parameters
  #new_params = optax.apply_updates(params, updates)

  #avg_distances[iters] = jnp.mean(final_rewards)
  interval = 100
  if (iters+1) % interval == 0:
    print(f"Iter: {iters}, Centroid Avg: {np.mean(avg_distances[iters+1-interval:iters+1]):.2f}, Centroid Std: {np.mean(std_distances[iters+1-interval:iters+1]):.2f}"  )


# Run and Plot

In [ ]:
#OLD CODE/COMMENTS
  #TRAPEZOID RULE TIMESTEP TO UPDATE KINEMATICS

  #masses at the halfway point
  #half_delta_m = 0.5*dt*mass_rates
  #masses += half_delta_m


  # Compute accelerations from forces and masses (F = ma -> a = F/m)
  #accelerations = forces / masses[..., None]  # Shape: (3, 2)

  #velocity at halfway point
  #half_delta_v = 0.5*dt*accelerations
  #velocities += half_delta_v

  #positions += velocities * dt #update posiitons using the halfway velocity

  #do the SECOND half of the time step for masses and velocities
  #masses += half_delta_m
  #velocities += half_delta_v

  #positions, velocities = trapezoid_timestep(positions, velocities, forces, masses, dt)

  #pumps =  UPDATE BASED ON NEURAL NETWORK
  #muscles = UPDATE BAASED ON NEURAL NETWORK


  #=== FANCIER WAY OF DOING FRICTION ===
  # This commented out block has a slightly fancier way of doing friciton where
  # the static friction opposes the force if the object is static but the force exceeds the max static force
  # I'm going to stick to the slightly simplified way which is like the original simulation.
  ##
  # Static friction force: balance external forces, but limited by the maximum static friction
  #static_friction_forces = jnp.where(
  #    force_magnitudes[..., None] <= static_friction_max[..., None],
  #    -forces,  # Fully balances the external force if within static friction limits
  #    -static_friction_max[..., None] * force_directions  # Maximum static friction force
  #)  # Shape: (batch_size, 3, 2)

  # Kinetic friction force: acts in the opposite direction of velocity
  #kinetic_friction_forces = -kinetic_friction_magnitudes[..., None] * velocity_directions  # Shape: (batch_size, 3, 2)

  # Combine static and kinetic friction forces
  #friction_forces = jnp.where(is_static[..., None], static_friction_forces, kinetic_friction_forces)

  # Compute the net forces: external forces + friction forces
  #net_forces = forces + friction_forces  # Shape: (batch_size, 3, 2)
  # Normalize the external forces for direction
  #force_directions = jnp.where(
  #    force_magnitudes[..., None] > epsilon,
  #    forces / force_magnitudes[..., None],
  #    0.0
  #)  # Shape: (batch_size, 3, 2)

  ##
  #Lets ignore pump forces for now...slightly annoying.
  # Compute pump forces based on conservation of momentum law :
  # F_pump = (dm/dt) * (fluid_velocity) = (dm/dt)*(leg_length / dt)
  #p_force_01 = mass_rates[...,0]*pump_v* dir_01  # Shape: (batch_size, 2)
  #p_force_02 = mass_rates[...,1]*pump_v* dir_02  # Shape: (batch_size, 2)
  #p_force_12 = mass_rates[...,2]*pump_v* dir_12  # Shape: (batch_size, 2)


In [ ]:
def compute_returns(rewards, gamma):
    """Compute the discounted rewards-to-go for each timestep in the trajectory."""
    returns = jnp.zeros_like(rewards)
    running_sum = 0
    for t in reversed(range(rewards.shape[0])):
        running_sum = rewards[t] + gamma * running_sum
        returns = returns.at[t].set(running_sum)
    return returns

def reinforce_update2(params, action_history, state_history, reward_history, gamma, optimizer_state, opt_update):
    """Update the parameters using the REINFORCE algorithm"""


    # Define a function to calculate the policy loss
    def policy_loss2(params, action_history, state_history, returns):
        # Compute the logits (output of the network)
        logits = forward(params, state_history)

        # Convert logits to action probabilities (softmax)
        action_probs = jax.nn.softmax(logits)

        # Gather the action probabilities for the taken actions
        action_taken_probs = action_probs[jnp.arange(action_history.shape[0]), action_history]

        # Compute the log-probabilities of the taken actions
        log_action_taken_probs = jnp.log(action_taken_probs)

        # Compute the loss (REINFORCE objective)
        loss = -jnp.sum(log_action_taken_probs * returns)

        return loss, action_probs

    # Initialize the gradients
    grad_fn = jax.value_and_grad(policy_loss2, has_aux=True)

    # Calculate the returns (reward-to-go) for each timestep
    returns = compute_returns(reward_history, gamma)


    # Compute the gradient of the loss function
    loss, action_probs = grad_fn(params, action_history, state_history, returns)

    # Apply the optimizer step (gradient update)
    updates, new_optimizer_state = opt_update(loss, optimizer_state)

    # Update the parameters
    new_params = optax.apply_updates(params, updates)

    return new_params, new_optimizer_state, loss


def reinforce_update(params, action_history, state_history, reward_history, log_action_taken_probs_history, gamma, optimizer_state, opt_update):
    """Update the parameters using the REINFORCE algorithm, given the log action probabilities."""

    # Calculate the returns (reward-to-go) for each timestep
    returns = compute_returns(reward_history, gamma)

    # Define a function to compute the REINFORCE loss
    def policy_loss(params, action_history, state_history, log_action_taken_probs_history, returns):
        # The loss function is the negative sum of log-probabilities weighted by returns
        loss = -jnp.sum(log_action_taken_probs_history * returns) / returns.shape[1]  # Normalize by batch size
        return loss

    # Compute the gradient of the loss function with respect to the parameters
    grad_fn = jax.value_and_grad(policy_loss)

    # Corrected: grad_fn returns BOTH loss and gradients
    loss, grads = grad_fn(params, action_history, state_history, log_action_taken_probs_history, returns)

    # Apply the optimizer step (gradient update)
    updates, new_optimizer_state = opt_update(grads, optimizer_state)

    # Update the model parameters
    new_params = optax.apply_updates(params, updates)

    return new_params, new_optimizer_state, loss